## Loading Datasets

In [1]:
import pandas as pd

df = pd.read_csv("/content/Churn Bank.csv")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   RowNumber        10000 non-null  int64  
 1   CustomerId       10000 non-null  int64  
 2   Surname          10000 non-null  object 
 3   CreditScore      10000 non-null  int64  
 4   Geography        10000 non-null  object 
 5   Gender           10000 non-null  object 
 6   Age              10000 non-null  int64  
 7   Tenure           10000 non-null  int64  
 8   Balance          10000 non-null  float64
 9   NumOfProducts    10000 non-null  int64  
 10  HasCrCard        10000 non-null  object 
 11  IsActiveMember   10000 non-null  int64  
 12  EstimatedSalary  10000 non-null  float64
 13  Exited           10000 non-null  int64  
dtypes: float64(2), int64(8), object(4)
memory usage: 1.1+ MB


In [2]:
df.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


There is no nedeed columns like "RowNumber", "CustomerId", "Surname", "Geography" (we deleted because just to learn)

And i know target predict is __Exited__ column

## Simple Cleaning Datasets <br>
+ __See duplicated dan null__

In [3]:
print(df.duplicated().sum())

df.isna().sum()

0


,0
RowNumber,0
CustomerId,0
Surname,0
CreditScore,0
Geography,0
Gender,0
Age,0
Tenure,0
Balance,0
NumOfProducts,0


There is no duplicated and null/missing value!
+ Drop column we no needed

In [4]:
df.drop(columns=["RowNumber", "CustomerId", "Surname", "Geography"], inplace=True)
df.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Female,43,2,125510.82,1,1,1,79084.10,0


+ __Check unique categorical column__ <br>
Make sure anomlies values

In [5]:
for col in df.select_dtypes("object").columns:
  print(f"{col} : {df[col].unique()}")

Gender : ['Female' 'Male' 'Male/u' 'Femalei' 'Male(']
HasCrCard : ['1' '0' "N'O"]


There is a anomaly values! We must delete!

In [17]:
# Replaced anomalies value in Gender column

df["Gender"] = df["Gender"].replace({
    "Male/u": "Male",
    "Femalei": "Female",
    "Male(": "Male"
})

df = df[df.HasCrCard != "N'O"] # We delete becouse this value isn't presenting other value

We make Male and Female value to 0 and 1

In [18]:
df["Gender"]= df["Gender"].apply(lambda x: 0 if x == "Male" else 1 if x == "Female" else 0)

<ipython-input-18-080608f7484c>:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["Gender"]= df["Gender"].apply(lambda x: 0 if x == "Male" else 1 if x == "Female" else 0)


See data after cleaning

In [19]:
df.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,1,42,2,0.00,1,1,1,101348.88,1
1,608,1,41,1,83807.86,1,0,1,112542.58,0
2,502,1,42,8,159660.80,3,1,0,113931.57,1
3,699,1,39,1,0.00,2,0,0,93826.63,0
4,850,1,43,2,125510.82,1,1,1,79084.10,0


### Splitting data to train and test

In [20]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

X = df[[col for col in df.columns if col != "Exited"]]
y = df["Exited"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

### Modeling <br>
I use LogisticRegression because its suitable for simple datasets and for learning

In [21]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

model = LogisticRegression()
model.fit(X_train, y_train)

y_pred_x = model.predict(X_train)
y_pred_y = model.predict(X_test)

acc_x = accuracy_score(y_train, y_pred_x)
acc_y = accuracy_score(y_test, y_pred_y)

print(f"train: {acc_x} | test: {acc_y}")

train: 0.7890986373296662 | test: 0.8045


/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


### Exporting model

In [22]:
import pickle
import os

path = "./model/model_out.pkl"
os.makedirs(os.path.dirname(path), exist_ok=True)

pickle_out = open(path, "wb")
pickle.dump(model, pickle_out)
pickle_out.close()